# IPI — data exploration starter

Environment note: packages live in `~/.local` (user site), the same environment the
`code/*.py` pipeline scripts run under. Nothing is installed in a separate venv, so
`import gigfilter` and friends work here exactly as they do in the scripts.

Storage is flat files — no database. Three tiers:

| tier | where | size | how to read |
|---|---|---|---|
| derived indices | `data/pilot/*.csv`, `docs/*.json` | KB | `pd.read_csv` |
| extracted price panel | `data/pilot/{balanced,recent,expanded}-prices.csv` | 4–85 MB | `pd.read_csv` |
| pipeline intermediates | `data/cdx-index/*.tsv` | 1.3–5.8 GB | **duckdb**, not pandas |
| raw archived pages | `data/pilot/html*/` | 86 GB | `gzip.open` one at a time |

In [ ]:
import os, sys, gzip, json, re
from pathlib import Path
import numpy as np, pandas as pd, duckdb

ROOT = Path("/home/exouser/IntelligencePriceIndex")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "code"))   # so `import gigfilter` works

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print(pd.__version__, duckdb.__version__)

## 1. Derived indices (tiny, git-tracked)

In [ ]:
ipi    = pd.read_csv("data/pilot/recent-ipi.csv")            # quarter, ipi
panel  = pd.read_csv("data/pilot/panel-ipi.csv")             # long historical panel
cats   = pd.read_csv("data/pilot/recent-category-indices-geks.csv")  # quarter x 7 categories

display(ipi.tail())
display(cats.tail())

In [ ]:
# the numbers frozen into the paper / website
paper = json.load(open("data/pilot/paper-numbers.json"))
site  = json.load(open("docs/data.json"))
print(sorted(paper)[:20])
print({k: site[k] for k in ("generated", "cadence", "base_period", "panel_gigs")})

## 2. The price panel

One row per (gig, archived snapshot date) with the three Fiverr package tiers.
`file_path` points back to the exact archived HTML the price was scraped from —
that is the lineage hook, use it whenever a number looks wrong.

In [ ]:
px = pd.read_csv("data/pilot/balanced-prices.csv")   # 292,447 rows, ~1s, ~106 MB resident
print(px.shape)
px.head(3)

In [ ]:
px["ym"] = px.year * 100 + px.month
(px.groupby("year")
   .agg(n=("price_basic", "size"),
        median_basic=("price_basic", "median"),
        sellers=("seller", "nunique"))
   .assign(median_basic=lambda d: d.median_basic.round(2)))

## 3. Big pipeline intermediates — use duckdb

`data/cdx-index/gig-month-index.tsv` is 1.3 GB and headerless; the classified /
deduped page tables are ~5.7 GB each. duckdb scans them out-of-core in seconds
where `pd.read_csv` would blow up memory.

In [ ]:
GMI = ("read_csv('data/cdx-index/gig-month-index.tsv', delim='\\t', header=false, "
       "columns={'gig_id':'VARCHAR','month':'VARCHAR','timestamp':'VARCHAR','category':'VARCHAR'})")

duckdb.sql(f"""
SELECT category, count(*) AS snapshots, count(DISTINCT gig_id) AS gigs
FROM {GMI}
GROUP BY 1 ORDER BY snapshots DESC
""").df()

In [ ]:
# coverage over time — 1.3 GB scanned in a few seconds
cov = duckdb.sql(f"""
SELECT month, count(DISTINCT gig_id) AS gigs
FROM {GMI}
GROUP BY 1 ORDER BY 1
""").df()

ax = cov.set_index("month").plot(figsize=(12, 3), legend=False,
                                 title="distinct gigs archived per month")
ax.set_xlabel("")
print(f"{len(cov)} months, {cov.month.min()}–{cov.month.max()}")

## 4. Raw archived pages

`data/pilot/html-{recent,balanced}/<seller>/<YYYYMMDD>_<slug>.html.gz`.
86 GB total — never glob the whole tree into memory; index it via the
download logs or the price panel's `file_path` instead.

In [ ]:
row  = px.iloc[0]
path = Path(row.file_path.replace(".html", ".html.gz"))
if not path.exists():                       # some waves store uncompressed paths
    path = Path(row.file_path)

html = gzip.open(path, "rt", errors="replace").read() if path.suffix == ".gz" else path.read_text(errors="replace")
print(path, f"{len(html):,} chars")
print(re.search(r"<title>(.*?)</title>", html, re.S).group(1)[:120])

In [ ]:
# the download logs are the cheap index into the HTML store
log = pd.read_csv("data/pilot/recent-download-log.tsv", sep="\t")
print(log.shape, log.status.value_counts().to_dict())
log.head(3)

## 5. Niche assignment (2026-08 event-study work)

In [ ]:
arr = pd.read_csv("data/pilot/niche-arrival.csv")
asg = pd.read_csv("data/pilot/niche-assignment.csv")
print(asg.shape, "usable:", int(asg.usable.sum()))
arr.dropna(subset=["arrival_quarter"]).sort_values("n_listings", ascending=False).head(10)